Взял из ямды normalized_embed построил по ним FAISS HNSW и сделал user вектор как по последним трекам, потом кажется он и помешал в норм качестве на обучении MLP, суть в том что эмбединги вообще не учитывают пользовательскеое поведение и по итогу был очень маленьке позитивное взаимодействие. Собрал ANN, потом на ANN кандидатах обучил CatBoost получлось впринципе нормально. Для MLP ранкера в базовом сетапе вообще было все плохо, добавил признаков получилось все равно плохо, но получше

In [1]:
!pip install -U pip
!pip install numpy pandas tqdm scikit-learn catboost datasets faiss-cpu
!pip install torch --index-url https://download.pytorch.org/whl/cu128

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Users\egorg\PycharmProjects\check5\.venv\Scripts\python.exe -m pip install -U pip



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Looking in indexes: https://download.pytorch.org/whl/cu128



[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from tqdm import tqdm

from check6.get_data import prepare
from check6.metrics import dcg_at_k, idcg_at_k
from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier, Pool
from datasets import load_from_disk
import faiss
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

C:\Users\egorg\PycharmProjects\check5\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import torch

print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("cuda version:", torch.version.cuda)

if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

torch version: 2.11.0+cu128
cuda available: True
cuda version: 12.8
gpu: NVIDIA GeForce RTX 4070 SUPER


Посмотрим структуру и соеденим

In [4]:
EMB_PATH = "yambda_embedings"

emb_ds = load_from_disk(EMB_PATH)
emb = emb_ds["train"]

emb_dim = len(emb[0]["normalized_embed"])

print("items:", len(emb))
print("dim:", emb_dim)

items: 7721749
dim: 128


In [5]:
EVENTS_PATH = "yambda"

data = prepare(EVENTS_PATH)

train_df = data["train_df"]
test_df = data["test_df"]
inter_df = data["inter_df"]
test_true = data["test_true"]

print("train_df:", train_df.shape)
print("test_df:", test_df.shape)
print("inter_df:", inter_df.shape)
print("test users:", len(test_true))

train_df: (14648478, 12)
test_df: (897978, 8)
inter_df: (4212276, 5)
test users: 2490


In [6]:
emb_items = set(emb["item_id"])

train_items = set(inter_df["item_id"].unique())
test_items = set(test_df["item_id"].unique())

train_in_emb = train_items & emb_items
test_in_emb = test_items & emb_items

print(f"emb items: {len(emb_items)}")

print(
    f"train items: {len(train_items)} "
    f"in embeddings: {len(train_in_emb)} "
    f"coverage: {len(train_in_emb) / len(train_items)}"
)

print(
    f"test items: {len(test_items)} | "
    f"in embeddings: {len(test_in_emb)} | "
    f"coverage: {len(test_in_emb) / len(test_items)}"
)

emb items: 7721749
train items: 470449 in embeddings: 431344 coverage: 0.9168772810655352
test items: 141609 | in embeddings: 135717 | coverage: 0.9583924750545516


Норм большое покрытие эмбеденгами почти нчиего не потеряли

In [7]:
candidate_items = train_in_emb

inter_df_emb = inter_df[inter_df["item_id"].isin(candidate_items)].copy()
train_df_emb = train_df[train_df["item_id"].isin(candidate_items)].copy()

test_true_emb = {
    uid: items & candidate_items
    for uid, items in test_true.items()
}

test_true_emb = {
    uid: items
    for uid, items in test_true_emb.items()
    if len(items) > 0
}

print(f"inter_df: {inter_df.shape} after {inter_df_emb.shape}")
print(f"train_df: {train_df.shape} after {train_df_emb.shape}")
print(f"test users: {len(test_true)} after {len(test_true_emb)}")

inter_df: (4212276, 5) after (4034293, 5)
train_df: (14648478, 12) after (14174502, 12)
test users: 2490 after 2489


In [8]:
emb_item_ids = emb["item_id"]

positions = [
    pos
    for pos, item_id in enumerate(emb_item_ids)
    if item_id in candidate_items
]

emb_part = emb.select(positions)

item_ids = np.array(emb_part["item_id"])
item_vecs = np.vstack(emb_part["normalized_embed"]).astype("float32")

item_id_to_pos = {
    item_id: pos
    for pos, item_id in enumerate(item_ids)
}

print(f"item_vecs: {item_vecs.shape}")
print(f"item_ids: {item_ids.shape}")
print(f"item_id_to_pos: {len(item_id_to_pos)}")

item_vecs: (431344, 128)
item_ids: (431344,)
item_id_to_pos: 431344


соберем проверим размеры

In [9]:
uids = np.array(inter_df_emb["uid"].drop_duplicates())
uid_to_pos = {
    uid: pos
    for pos, uid in enumerate(uids)
}

dim = item_vecs.shape[1]

user_vecs = np.zeros((len(uids), dim), dtype="float32")
user_weights = np.zeros(len(uids), dtype="float32")

for uid, item_id, w in inter_df_emb[["uid", "item_id", "w"]].itertuples(index=False):
    u_pos = uid_to_pos[uid]
    i_pos = item_id_to_pos[item_id]

    user_vecs[u_pos] += float(w) * item_vecs[i_pos]
    user_weights[u_pos] += float(w)

user_vecs = user_vecs / np.maximum(user_weights[:, None], 1e-12)

user_vecs = user_vecs / np.maximum(
    np.linalg.norm(user_vecs, axis=1, keepdims=True),
    1e-12,
)

print(f"user_vecs: {user_vecs.shape}")
print(f"uids: {uids.shape}")
print(f"zero-weight users: {(user_weights == 0).sum()}")

user_vecs: (2496, 128)
uids: (2496,)
zero-weight users: 0


In [10]:
dim = item_vecs.shape[1]

index = faiss.IndexHNSWFlat(dim, 32, faiss.METRIC_INNER_PRODUCT)
index.hnsw.efConstruction = 200
index.hnsw.efSearch = 128

index.add(item_vecs)

print(f"index size: {index.ntotal}")
print(f"dim: {dim}")

index size: 431344
dim: 128


In [11]:
seen = inter_df_emb.groupby("uid")["item_id"].apply(set).to_dict()

def ann_recommend(uid, k=10, candidate_k=300):
    if uid not in uid_to_pos:
        return []

    u_pos = uid_to_pos[uid]
    query = user_vecs[u_pos:u_pos + 1]

    scores, positions = index.search(query, candidate_k)

    seen_items = seen.get(uid, set())
    recs = []

    for pos in positions[0]:
        if pos < 0:
            continue

        item_id = item_ids[pos]

        if item_id in seen_items:
            continue

        recs.append(int(item_id))

        if len(recs) >= k:
            break

    return recs


example_uid = uids[0]
recs = ann_recommend(example_uid, k=10, candidate_k=300)

print(f"uid: {example_uid}")
print(f"recs: {recs}")
print(f"n_recs: {len(recs)}")

uid: 600
recs: [3113777, 8085577, 7085335, 7362974, 6164286, 405511, 7704958, 6847363, 3672820, 8594591]
n_recs: 10


In [12]:
hnsw_index = index

flat_index = faiss.IndexFlatIP(item_vecs.shape[1])
flat_index.add(item_vecs)

def eval_faiss_index(index_obj, name, top_k=10, candidate_k=300):
    hits = 0
    total = 0
    precision_hits = 0
    users = 0
    ndcgs = []

    eval_uids = [uid for uid in test_true_emb if uid in uid_to_pos]

    for uid in eval_uids:
        u_pos = uid_to_pos[uid]
        query = user_vecs[u_pos:u_pos + 1]

        scores, positions = index_obj.search(query, candidate_k)

        seen_items = seen.get(uid, set())
        true_items = test_true_emb[uid]

        recs = []

        for pos in positions[0]:
            item_id = int(item_ids[pos])

            if item_id in seen_items:
                continue

            recs.append(item_id)

            if len(recs) >= top_k:
                break

        if len(recs) == 0:
            continue

        recs_set = set(recs)

        hits += len(recs_set & true_items)
        precision_hits += len(recs_set & true_items)
        total += len(true_items)
        users += 1

        dcg = dcg_at_k(recs, true_items, top_k)
        idcg = idcg_at_k(true_items, top_k)

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    recall = hits / total if total > 0 else 0
    precision = precision_hits / (users * top_k) if users > 0 else 0
    ndcg = np.mean(ndcgs) if ndcgs else 0

    print(f"{name} Recall@{top_k}: {recall:.6f}")
    print(f"{name} Precision@{top_k}: {precision:.6f}")
    print(f"{name} NDCG@{top_k}: {ndcg:.6f}")


eval_faiss_index(hnsw_index, "HNSW", top_k=10, candidate_k=300)
eval_faiss_index(flat_index, "FlatIP", top_k=10, candidate_k=300)

HNSW Recall@10: 0.000373
HNSW Precision@10: 0.008035
HNSW NDCG@10: 0.008209
FlatIP Recall@10: 0.000375
FlatIP Precision@10: 0.008076
FlatIP NDCG@10: 0.008253


качество одиннаковое оставил HNSW

In [13]:
def build_user_embeddings(
    inter_df,
    item_vecs,
    item_id_to_pos,
    last_n=50,
):
    df = inter_df.copy()

    df = (
        df.sort_values(["uid", "ts"])
        .groupby("uid", group_keys=False)
        .tail(last_n)
    )

    uids = np.array(df["uid"].drop_duplicates())

    uid_to_pos = {
        uid: pos
        for pos, uid in enumerate(uids)
    }

    user_vecs = np.zeros((len(uids), item_vecs.shape[1]), dtype="float32")
    user_weights = np.zeros(len(uids), dtype="float32")

    for uid, item_id, w in df[["uid", "item_id", "w"]].itertuples(index=False):
        if item_id not in item_id_to_pos:
            continue

        u_pos = uid_to_pos[uid]
        i_pos = item_id_to_pos[item_id]

        user_vecs[u_pos] += float(w) * item_vecs[i_pos]
        user_weights[u_pos] += float(w)

    user_vecs = user_vecs / np.maximum(user_weights[:, None], 1e-12)

    user_vecs = user_vecs / np.maximum(
        np.linalg.norm(user_vecs, axis=1, keepdims=True),
        1e-12,
    )

    return {
        "uids": uids,
        "uid_to_pos": uid_to_pos,
        "user_vecs": user_vecs,
        "user_weights": user_weights,
        "last_n": last_n,
    }


def make_seen(inter_df):
    return inter_df.groupby("uid")["item_id"].apply(set).to_dict()


def make_rank_df(
    index,
    item_ids,
    user_pack,
    users,
    seen,
    true_dict,
    candidate_k=300,
    search_k=1000,
):
    user_vecs = user_pack["user_vecs"]
    uid_to_pos = user_pack["uid_to_pos"]

    rows = []

    for uid in users:
        if uid not in uid_to_pos:
            continue

        u_pos = uid_to_pos[uid]
        query = user_vecs[u_pos:u_pos + 1]

        scores, positions = index.search(query, search_k)

        seen_items = seen.get(uid, set())
        true_items = true_dict.get(uid, set())

        rank = 0

        for score, pos in zip(scores[0], positions[0]):
            if pos < 0:
                continue

            item_id = int(item_ids[pos])

            if item_id in seen_items:
                continue

            rows.append({
                "uid": uid,
                "item_id": item_id,
                "ann_score": float(score),
                "ann_rank": rank,
                "user_pos": int(u_pos),
                "item_pos": int(pos),
                "label": int(item_id in true_items),
            })

            rank += 1

            if rank >= candidate_k:
                break

    return pd.DataFrame(rows)

In [14]:
def eval_rank_df(
    rank_df,
    true_dict,
    score_col="ann_score",
    k=10,
    users=None,
):
    if users is None:
        users = rank_df["uid"].unique()

    hits = 0
    total = 0
    precision_hits = 0
    n_users = 0
    ndcgs = []

    for uid in users:
        if uid not in true_dict:
            continue

        cur = rank_df[rank_df["uid"] == uid]

        if len(cur) == 0:
            continue

        recs = (
            cur.sort_values(score_col, ascending=False)
            ["item_id"]
            .head(k)
            .astype(int)
            .tolist()
        )

        true_items = true_dict[uid]
        hit_count = len(set(recs) & true_items)

        hits += hit_count
        precision_hits += hit_count
        total += len(true_items)
        n_users += 1

        dcg = dcg_at_k(recs, true_items, k)
        idcg = idcg_at_k(true_items, k)

        if idcg > 0:
            ndcgs.append(dcg / idcg)

    recall = hits / total if total > 0 else 0
    precision = precision_hits / (n_users * k) if n_users > 0 else 0
    ndcg = np.mean(ndcgs) if ndcgs else 0

    return {
        "recall": recall,
        "precision": precision,
        "ndcg": ndcg,
        "users": n_users,
        "hits": hits,
        "total": total,
    }


def print_metrics(name, metrics, k=10):
    print(f"{name} Recall@{k}: {metrics['recall']:.6f}")
    print(f"{name} Precision@{k}: {metrics['precision']:.6f}")
    print(f"{name} NDCG@{k}: {metrics['ndcg']:.6f}")
    print(f"users: {metrics['users']} | hits: {metrics['hits']} / {metrics['total']}")

так как user эмбедингов нет пользовательский эмбэдинг был построен как взвешенное среднее от треков из истории пользователя, получались слишком длинные вектора поэтому обрезал до 50 ти полследних, (подбирал это значечние) пробовал усреднять все но получалось хуже.

In [15]:
LAST_N = 50
CANDIDATE_K = 300
SEARCH_K = 1000

seen = make_seen(inter_df_emb)

user_pack = build_user_embeddings(
    inter_df=inter_df_emb,
    item_vecs=item_vecs,
    item_id_to_pos=item_id_to_pos,
    last_n=LAST_N,
)

rank_users = list(test_true_emb.keys())

rank_df = make_rank_df(
    index=index,
    item_ids=item_ids,
    user_pack=user_pack,
    users=rank_users,
    seen=seen,
    true_dict=test_true_emb,
    candidate_k=CANDIDATE_K,
    search_k=SEARCH_K,
)

print(f"LAST_N: {LAST_N}")
print(f"CANDIDATE_K: {CANDIDATE_K}")
print(f"rank_df: {rank_df.shape}")
print(f"users: {rank_df['uid'].nunique()}")
print(f"positive rate: {rank_df['label'].mean():.6f}")
print(f"positives: {rank_df['label'].sum()}")

LAST_N: 50
CANDIDATE_K: 300
rank_df: (746356, 7)
users: 2489
positive rate: 0.007184
positives: 5362


In [16]:
ann_metrics = eval_rank_df(
    rank_df=rank_df,
    true_dict=test_true_emb,
    score_col="ann_score",
    k=10,
)

print_metrics("ANN", ann_metrics, k=10)

ANN Recall@10: 0.000507
ANN Precision@10: 0.010928
ANN NDCG@10: 0.011578
users: 2489 | hits: 272 / 536591


In [17]:
def make_xy(rank_part, user_pack, item_vecs):
    user_vecs = user_pack["user_vecs"]

    user_part = user_vecs[rank_part["user_pos"].to_numpy()]
    item_part = item_vecs[rank_part["item_pos"].to_numpy()]
    ann_part = rank_part[["ann_score", "ann_rank"]].to_numpy(dtype="float32")

    x = np.hstack([
        ann_part,
        user_part,
        item_part,
    ]).astype("float32")

    y = rank_part["label"].to_numpy(dtype="int32")

    return x, y


def make_feature_names(user_dim, item_dim):
    return (
        ["ann_score", "ann_rank"]
        + [f"u_emb_{i}" for i in range(user_dim)]
        + [f"i_emb_{i}" for i in range(item_dim)]
    )


def train_catboost(
    rank_df,
    user_pack,
    item_vecs,
    params=None,
    valid_size=0.2,
):
    if params is None:
        params = {}

    train_uids, valid_uids = train_test_split(
        rank_df["uid"].unique(),
        test_size=valid_size,
        random_state=RANDOM_STATE,
    )

    train_part = rank_df[rank_df["uid"].isin(train_uids)]
    valid_part = rank_df[rank_df["uid"].isin(valid_uids)]

    x_train, y_train = make_xy(train_part, user_pack, item_vecs)
    x_valid, y_valid = make_xy(valid_part, user_pack, item_vecs)

    feature_names = make_feature_names(
        user_dim=user_pack["user_vecs"].shape[1],
        item_dim=item_vecs.shape[1],
    )

    pos = y_train.sum()
    neg = len(y_train) - pos

    train_pool = Pool(x_train, y_train, feature_names=feature_names)
    valid_pool = Pool(x_valid, y_valid, feature_names=feature_names)

    model_params = {
        "iterations": 500,
        "depth": 6,
        "learning_rate": 0.05,
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "scale_pos_weight": neg / max(pos, 1),
        "random_seed": RANDOM_STATE,
        "verbose": 100,
    }

    model_params.update(params)

    model = CatBoostClassifier(**model_params)

    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    info = {
        "train_uids": train_uids,
        "valid_uids": valid_uids,
        "feature_names": feature_names,
        "train_rows": len(y_train),
        "valid_rows": len(y_valid),
        "train_positives": int(y_train.sum()),
        "valid_positives": int(y_valid.sum()),
    }

    return model, info


def score_catboost(rank_df, model, user_pack, item_vecs, batch_size=100_000):
    result = rank_df.copy()
    scores = np.zeros(len(result), dtype="float32")

    for start in range(0, len(result), batch_size):
        end = min(start + batch_size, len(result))
        part = result.iloc[start:end]

        x_part, _ = make_xy(part, user_pack, item_vecs)
        scores[start:end] = model.predict_proba(x_part)[:, 1]

    result["cat_score"] = scores

    return result

In [18]:
cat_model, cat_info = train_catboost(
    rank_df=rank_df,
    user_pack=user_pack,
    item_vecs=item_vecs,
)

print(f"train rows: {cat_info['train_rows']}")
print(f"valid rows: {cat_info['valid_rows']}")
print(f"train positives: {cat_info['train_positives']}")
print(f"valid positives: {cat_info['valid_positives']}")

0:	test: 0.5802738	best: 0.5802738 (0)	total: 243ms	remaining: 2m 1s
100:	test: 0.6303211	best: 0.6304506 (99)	total: 9.79s	remaining: 38.7s
200:	test: 0.6408031	best: 0.6427474 (150)	total: 19.3s	remaining: 28.7s
300:	test: 0.6447265	best: 0.6452128 (298)	total: 28.3s	remaining: 18.7s
400:	test: 0.6396140	best: 0.6453779 (308)	total: 36.8s	remaining: 9.07s
499:	test: 0.6379689	best: 0.6453779 (308)	total: 45s	remaining: 0us

bestTest = 0.6453778632
bestIteration = 308

Shrink model to first 309 iterations.
train rows: 596956
valid rows: 149400
train positives: 4466
valid positives: 896


In [19]:
rank_df_cat = score_catboost(
    rank_df=rank_df,
    model=cat_model,
    user_pack=user_pack,
    item_vecs=item_vecs,
)

cat_metrics = eval_rank_df(
    rank_df=rank_df_cat,
    true_dict=test_true_emb,
    score_col="cat_score",
    k=10,
    users=cat_info["valid_uids"],
)

print_metrics("CatBoost", cat_metrics, k=10)

CatBoost Recall@10: 0.000943
CatBoost Precision@10: 0.018675
CatBoost NDCG@10: 0.021294
users: 498 | hits: 93 / 98640


In [20]:
fi = pd.DataFrame({
    "feature": cat_info["feature_names"],
    "importance": cat_model.get_feature_importance(),
}).sort_values("importance", ascending=False)

print(f"features: {len(cat_info['feature_names'])}")
print(f"importance values: {len(cat_model.get_feature_importance())}")
print(f"fi shape: {fi.shape}")

display(fi.head(30))

features: 258
importance values: 258
fi shape: (258, 2)


,feature,importance
0,ann_score,3.764845
83,u_emb_81,2.577609
202,i_emb_72,1.986913
247,i_emb_117,1.616184
14,u_emb_12,1.311584
77,u_emb_75,1.260634
18,u_emb_16,1.246023
218,i_emb_88,1.215760
208,i_emb_78,1.204742
56,u_emb_54,1.055151


In [21]:
param_grid = [
    {"depth": 4, "learning_rate": 0.05, "iterations": 400, "verbose": False},
    {"depth": 6, "learning_rate": 0.05, "iterations": 400, "verbose": False},
    {"depth": 6, "learning_rate": 0.03, "iterations": 600, "verbose": False},
    {"depth": 8, "learning_rate": 0.03, "iterations": 600, "verbose": False},
]

tuning_results = []

best_model = None
best_info = None
best_ndcg = -1
best_params = None

for params in param_grid:
    model, info = train_catboost(
        rank_df=rank_df,
        user_pack=user_pack,
        item_vecs=item_vecs,
        params=params,
    )

    scored = score_catboost(
        rank_df=rank_df,
        model=model,
        user_pack=user_pack,
        item_vecs=item_vecs,
    )

    metrics = eval_rank_df(
        rank_df=scored,
        true_dict=test_true_emb,
        score_col="cat_score",
        k=10,
        users=info["valid_uids"],
    )

    row = {
        **params,
        "recall@10": metrics["recall"],
        "precision@10": metrics["precision"],
        "ndcg@10": metrics["ndcg"],
    }

    tuning_results.append(row)

    if metrics["ndcg"] > best_ndcg:
        best_ndcg = metrics["ndcg"]
        best_model = model
        best_info = info
        best_params = params

tuning_results = pd.DataFrame(tuning_results)
tuning_results

,depth,learning_rate,iterations,verbose,recall@10,precision@10,ndcg@10
0,4,0.05,400,False,0.000821,0.016265,0.018765
1,6,0.05,400,False,0.000943,0.018675,0.021294
2,6,0.03,600,False,0.001125,0.022289,0.029096
3,8,0.03,600,False,0.001237,0.024498,0.031244


In [22]:
rank_df_best_cat = score_catboost(
    rank_df=rank_df,
    model=best_model,
    user_pack=user_pack,
    item_vecs=item_vecs,
)

best_cat_metrics = eval_rank_df(
    rank_df=rank_df_best_cat,
    true_dict=test_true_emb,
    score_col="cat_score",
    k=10,
    users=best_info["valid_uids"],
)

print(f"best params: {best_params}")
print_metrics("Best CatBoost", best_cat_metrics, k=10)

best params: {'depth': 8, 'learning_rate': 0.03, 'iterations': 600, 'verbose': False}
Best CatBoost Recall@10: 0.001237
Best CatBoost Precision@10: 0.024498
Best CatBoost NDCG@10: 0.031244
users: 498 | hits: 122 / 98640


такой сетап побил прошлый чекпоинт не сильно, но побил

In [31]:
def make_mlp_xy(rank_part, user_pack, item_vecs, max_rank):
    user_vecs = user_pack["user_vecs"]

    user_part = user_vecs[rank_part["user_pos"].to_numpy()]
    item_part = item_vecs[rank_part["item_pos"].to_numpy()]

    interaction_part = user_part * item_part
    diff_part = np.abs(user_part - item_part)

    ann_score = rank_part["ann_score"].to_numpy(dtype="float32").reshape(-1, 1)

    ann_rank = rank_part["ann_rank"].to_numpy(dtype="float32").reshape(-1, 1)
    ann_rank = ann_rank / max_rank

    x = np.hstack([
        ann_score,
        ann_rank,
        user_part,
        item_part,
        interaction_part,
        diff_part,
    ]).astype("float32")

    y = rank_part["label"].to_numpy(dtype="float32")

    return x, y


max_rank = rank_df["ann_rank"].max()

mlp_train_part = rank_df[rank_df["uid"].isin(cat_info["train_uids"])]
mlp_valid_part = rank_df[rank_df["uid"].isin(cat_info["valid_uids"])]

x_train_mlp, y_train_mlp = make_mlp_xy(
    mlp_train_part,
    user_pack,
    item_vecs,
    max_rank=max_rank,
)

x_valid_mlp, y_valid_mlp = make_mlp_xy(
    mlp_valid_part,
    user_pack,
    item_vecs,
    max_rank=max_rank,
)

print(f"x_train: {x_train_mlp.shape}")
print(f"x_valid: {x_valid_mlp.shape}")
print(f"train positives: {int(y_train_mlp.sum())}")
print(f"valid positives: {int(y_valid_mlp.sum())}")

x_train: (596956, 514)
x_valid: (149400, 514)
train positives: 4466
valid positives: 896


In [32]:
device = torch.device("cuda")

if device.type == "cuda":
    print(f"gpu: {torch.cuda.get_device_name(0)}")

gpu: NVIDIA GeForce RTX 4070 SUPER


In [33]:
class MLPRanker(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),

            nn.Linear(128, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


batch_size = 4096

train_ds = TensorDataset(
    torch.tensor(x_train_mlp, dtype=torch.float32),
    torch.tensor(y_train_mlp, dtype=torch.float32),
)

valid_ds = TensorDataset(
    torch.tensor(x_valid_mlp, dtype=torch.float32),
    torch.tensor(y_valid_mlp, dtype=torch.float32),
)

train_loader = DataLoader(
    train_ds,
    batch_size=batch_size,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=batch_size,
    shuffle=False,
    pin_memory=(device.type == "cuda"),
)

mlp_model = MLPRanker(input_dim=x_train_mlp.shape[1]).to(device)

pos = y_train_mlp.sum()
neg = len(y_train_mlp) - pos

pos_weight = torch.tensor(
    np.sqrt(neg / max(pos, 1)),
    dtype=torch.float32,
).to(device)

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

optimizer = torch.optim.AdamW(
    mlp_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

print(f"device: {device}")
print(f"batch_size: {batch_size}")
print(f"train batches: {len(train_loader)}")
print(f"valid batches: {len(valid_loader)}")
print(f"pos_weight: {pos_weight.item():.2f}")
print(f"params: {sum(p.numel() for p in mlp_model.parameters()):,}")

device: cuda
batch_size: 4096
train batches: 146
valid batches: 37
pos_weight: 11.52
params: 165,633


In [34]:
EPOCHS = 15

best_valid_loss = float("inf")
best_mlp_state = None
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    mlp_model.train()
    train_loss = 0.0

    train_pbar = tqdm(
        train_loader,
        desc=f"epoch {epoch}/{EPOCHS} train",
        leave=False,
    )

    for x_batch, y_batch in train_pbar:
        x_batch = x_batch.to(device, non_blocking=True)
        y_batch = y_batch.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = mlp_model(x_batch)
        loss = loss_fn(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(x_batch)
        train_pbar.set_postfix(loss=f"{loss.item():.4f}")

    train_loss /= len(train_ds)

    mlp_model.eval()
    valid_loss = 0.0

    valid_pbar = tqdm(
        valid_loader,
        desc=f"epoch {epoch}/{EPOCHS} valid",
        leave=False,
    )

    with torch.no_grad():
        for x_batch, y_batch in valid_pbar:
            x_batch = x_batch.to(device, non_blocking=True)
            y_batch = y_batch.to(device, non_blocking=True)

            logits = mlp_model(x_batch)
            loss = loss_fn(logits, y_batch)

            valid_loss += loss.item() * len(x_batch)
            valid_pbar.set_postfix(loss=f"{loss.item():.4f}")

    valid_loss /= len(valid_ds)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_epoch = epoch
        best_mlp_state = {
            k: v.detach().cpu().clone()
            for k, v in mlp_model.state_dict().items()
        }

    print(
        f"epoch {epoch} | "
        f"train_loss: {train_loss:.6f} | "
        f"valid_loss: {valid_loss:.6f}"
    )

mlp_model.load_state_dict(best_mlp_state)
mlp_model = mlp_model.to(device)

print(f"best epoch: {best_epoch}")
print(f"best valid_loss: {best_valid_loss:.6f}")

epoch 1 | train_loss: 0.773012 | valid_loss: 0.714475


epoch 2 | train_loss: 0.723972 | valid_loss: 0.676452


epoch 3 | train_loss: 0.673740 | valid_loss: 0.620748


epoch 4 | train_loss: 0.607886 | valid_loss: 0.537523


epoch 5 | train_loss: 0.534194 | valid_loss: 0.460928


...

epoch 11 | train_loss: 0.296274 | valid_loss: 0.258377


epoch 12 | train_loss: 0.285656 | valid_loss: 0.255604


epoch 13 | train_loss: 0.278470 | valid_loss: 0.251840


epoch 14 | train_loss: 0.270141 | valid_loss: 0.250558


epoch 15 | train_loss: 0.266012 | valid_loss: 0.251164
best epoch: 14
best valid_loss: 0.250558


In [35]:
def score_mlp(rank_df, mlp_model, user_pack, item_vecs, max_rank, batch_size=100_000):
    result = rank_df.copy()
    scores = np.zeros(len(result), dtype="float32")

    mlp_model.eval()

    with torch.no_grad():
        for start in range(0, len(result), batch_size):
            end = min(start + batch_size, len(result))
            part = result.iloc[start:end]

            x_part, _ = make_mlp_xy(
                part,
                user_pack,
                item_vecs,
                max_rank=max_rank,
            )

            x_part = torch.tensor(x_part, dtype=torch.float32).to(device)

            logits = mlp_model(x_part)
            probs = torch.sigmoid(logits).detach().cpu().numpy()

            scores[start:end] = probs

    result["mlp_score"] = scores

    return result


rank_df_mlp = score_mlp(
    rank_df=rank_df,
    mlp_model=mlp_model,
    user_pack=user_pack,
    item_vecs=item_vecs,
    max_rank=max_rank,
)

mlp_metrics = eval_rank_df(
    rank_df=rank_df_mlp,
    true_dict=test_true_emb,
    score_col="mlp_score",
    k=10,
    users=cat_info["valid_uids"],
)

print_metrics("MLP", mlp_metrics, k=10)

MLP Recall@10: 0.000862
MLP Precision@10: 0.017068
MLP NDCG@10: 0.018553
users: 498 | hits: 85 / 98640


По итогу не получилось побить CatBoost, что то совсем плохое качество баловался с параметрами обучения очень быстро модель переобучалась, увеличивал количество признаков добавил еще произведение. Проблема наверняка в user векторе